# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from datasets import load_dataset
from itertools import islice
import pandas as pd

# Load FlyRank warehouse sample
daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

# Use the first 5000 rows
sample = pd.DataFrame(list(islice(daily, 5000)))

print("Dataset Shape:", sample.shape)
sample.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset Shape: (5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


# Ranked Actions + Reason Codes

The purpose of this playbook is to convert the model output into a ranked list of content review actions.

Pages are prioritized using a simple priority score derived from historical search performance. Each recommendation is accompanied by a reason code so that reviewers understand why a page appears in the queue.

## Archetype → Action Mapping

| Content Archetype | Recommended Action | Reason Code |
|-------------------|-------------------|-------------|
| High impressions with poor ranking | Refresh content and improve topical coverage | RC-01 |
| High impressions with very low clicks | Review title tag and meta description | RC-02 |
| Moderate search performance | Monitor before taking action | RC-03 |
| Low visibility and low activity | No immediate action | RC-04 |

These recommendations are intended to support content teams rather than replace human judgement.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

queue = sample.copy()

queue["priority_score"] = (
    queue["gsc_impressions"] / 10
    - queue["gsc_clicks"]
)

def reason_code(score):
    if score >= queue["priority_score"].quantile(0.75):
        return "RC-01"
    elif score >= queue["priority_score"].median():
        return "RC-02"
    elif score >= queue["priority_score"].quantile(0.25):
        return "RC-03"
    return "RC-04"

queue["reason_code"] = queue["priority_score"].apply(reason_code)

action_map = {
    "RC-01": "Refresh content",
    "RC-02": "Improve CTR elements",
    "RC-03": "Monitor performance",
    "RC-04": "No immediate action"
}

queue["recommended_action"] = queue["reason_code"].map(action_map)

ranked_queue = queue.sort_values(
    "priority_score",
    ascending=False
)

ranked_queue[
    [
        "client_hash_id",
        "content_hash_id",
        "priority_score",
        "reason_code",
        "recommended_action"
    ]
].head(20)

,client_hash_id,content_hash_id,priority_score,reason_code,recommended_action
3825,client_9958f0a7ae1df715,content_213eb91f21a43550,42.4,RC-01,Refresh content
446,client_9958f0a7ae1df715,content_f94fe855380e150f,27.3,RC-01,Refresh content
1221,client_9958f0a7ae1df715,content_f94fe855380e150f,25.5,RC-01,Refresh content
146,client_9958f0a7ae1df715,content_f94fe855380e150f,20.6,RC-01,Refresh content
3884,client_9958f0a7ae1df715,content_f94fe855380e150f,19.7,RC-01,Refresh content
766,client_9958f0a7ae1df715,content_f94fe855380e150f,19.0,RC-01,Refresh content
1019,client_9958f0a7ae1df715,content_f94fe855380e150f,18.7,RC-01,Refresh content
1537,client_9958f0a7ae1df715,content_f5950be18c9f27db,16.5,RC-01,Refresh content
349,client_9958f0a7ae1df715,content_d02be57d816cf3d7,16.4,RC-01,Refresh content
953,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,16.2,RC-01,Refresh content


# Intended Use and Limits

This playbook is intended for SEO analysts and content teams to prioritize manual review of pages that may benefit from optimization.

The recommendations are based on historical search performance signals and are intended as decision-support.

The playbook should not be interpreted as proof that refreshing a page will improve rankings or traffic.

The recommendations are suitable for development and research purposes and should be reviewed before production use.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue["recommended_action"].value_counts()

,count
recommended_action,
Improve CTR elements,1454
Refresh content,1312
Monitor performance,1246
No immediate action,988


# Human Review

Every recommendation should be reviewed before any changes are made.

Reviewers should verify:

- Search intent still matches user needs.
- Content quality and accuracy.
- Technical SEO issues.
- Internal linking opportunities.
- Business priorities and seasonality.

## No-Go List

The following actions should never be automated:

- Publishing rewritten content.
- Deleting pages.
- Redirecting URLs.
- Changing page titles automatically.
- Changing structured data automatically.
- Deploying recommendations without editorial review.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review_queue = ranked_queue[
    [
        "reason_code",
        "recommended_action",
        "priority_score"
    ]
].head(20)

review_queue

,reason_code,recommended_action,priority_score
3825,RC-01,Refresh content,42.4
446,RC-01,Refresh content,27.3
1221,RC-01,Refresh content,25.5
146,RC-01,Refresh content,20.6
3884,RC-01,Refresh content,19.7
766,RC-01,Refresh content,19.0
1019,RC-01,Refresh content,18.7
1537,RC-01,Refresh content,16.5
349,RC-01,Refresh content,16.4
953,RC-01,Refresh content,16.2


# Monitoring and Retraining

The recommendation system should be reviewed regularly.

Potential retraining triggers include:

- A noticeable decline in model performance.
- Significant changes in search behaviour.
- Availability of additional historical data.
- Changes in Google Search Console metrics.
- Changes to the content strategy.

Recommendations should continue to be compared against the baseline rule before replacing it.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring = pd.DataFrame({

    "Trigger":[
        "Performance decline",
        "New historical data",
        "Search behaviour changes",
        "Quarterly review"
    ],

    "Recommended Response":[
        "Retrain model",
        "Rebuild feature set",
        "Re-evaluate baseline",
        "Audit recommendations"
    ]

})

monitoring

,Trigger,Recommended Response
0,Performance decline,Retrain model
1,New historical data,Rebuild feature set
2,Search behaviour changes,Re-evaluate baseline
3,Quarterly review,Audit recommendations


# Exports for the Paper

The ranked recommendation queue is exported to the work/outputs directory.

This export supports the final research paper and can be regenerated whenever the notebook is executed.

The exported file should be treated as a research artifact rather than a production recommendation system.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

output_file = "work/outputs/action_playbook_queue.csv"

ranked_queue.to_csv(
    output_file,
    index=False
)

print("Queue exported successfully.")
print(output_file)

Queue exported successfully.
work/outputs/action_playbook_queue.csv


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Ranked actions, reason codes, intended use, limits, human review, monitoring, and exports are included.
- [x] The queue is exported to `work/outputs/`.
- [x] Committed to my repo under `work/notebooks/`, then submitted my public repository URL.